In [0]:
%sql

-- Build the final monthly inference dataset used for prediction.
--
-- This table combines:
-- - monthly consumption targets by `group_id` and `timestamp_utc`
-- - static group-level metadata
-- - calendar-derived features
--
-- Join logic:
-- - consumption is the base grain of the dataset:
--   one row per `group_id` and `timestamp_utc`
-- - group metadata is joined by `group_id`
-- - calendar features are joined by `timestamp_utc`
--
-- Final output grain:
-- - one row per `group_id` and `timestamp_utc`

CREATE OR REPLACE TABLE fortum_challenge_data.03_gold.monthly_inference_dataset AS

WITH consumption AS (
    SELECT
        timestamp_utc,
        group_id,
        target_consumption
    FROM fortum_challenge_data.02_silver.consumption_monthly_inference_clean
),

group_data AS (
    SELECT
        group_id,
        macro_region,
        region,
        municipality,
        segment,
        product_type,
        consumption_bucket
    FROM fortum_challenge_data.02_silver.group_data_clean
),

calendar AS (
    SELECT
        timestamp_utc,
        is_weekend,
        is_holiday,
        working_day,
        desc,
        hour_of_day,
        day_of_week,
        month,
        day_of_year
    FROM fortum_challenge_data.02_silver.calendar_monthly_inference_clean
),

joined AS (
    SELECT
        c.timestamp_utc,
        c.group_id,
        c.target_consumption,
        g.macro_region,
        g.region,
        g.municipality,
        g.segment,
        g.product_type,
        g.consumption_bucket,
        cal.working_day,
        cal.desc,
        cal.is_weekend,
        cal.is_holiday,
        cal.hour_of_day,
        cal.day_of_week,
        cal.month,
        cal.day_of_year
    FROM consumption c
    LEFT JOIN group_data g
        ON c.group_id = g.group_id
    LEFT JOIN calendar cal
        ON c.timestamp_utc = cal.timestamp_utc
)

SELECT *
FROM joined
ORDER BY group_id, timestamp_utc;